# SFT Direct 50-Tool Evaluation

Evaluate the SFT adapter on unseen direct requests covering the 50-tool GRPO curriculum. Each case provides only its assigned 10-tool system prompt batch.

In [1]:
from dataclasses import asdict
from pathlib import Path
import csv
import json
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from toolcall_rl.evaluation.direct_50_cases import DIRECT_50_EVAL_CASES
from toolcall_rl.evaluation.scoring import score_response

MODEL_ID = "HuggingFaceTB/SmolLM-1.7B-Instruct"
ADAPTER_DIR = PROJECT_ROOT / "outputs" / "sft_smollm_20tools"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

len(DIRECT_50_EVAL_CASES), str(ADAPTER_DIR)

(50, '/home/shubeeksh/projects/toolcall-rl/outputs/sft_smollm_20tools')

## Load SFT Adapter

In [2]:
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32
tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
base_model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=dtype)
model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
model.to(device)
model.eval()
device

/home/shubeeksh/projects/toolcall-rl/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 218/218 [00:01<00:00, 214.91it/s]


'cuda'

## Generate And Score

In [3]:
def render_messages(messages):
    if tokenizer.chat_template:
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return "\n".join([*(f"<|{m['role']}|>\n{m['content']}" for m in messages), "<|assistant|>\n"])


def generate_response(case, max_new_tokens=220):
    messages = [
        {"role": "system", "content": case.system_prompt},
        {"role": "user", "content": case.prompt},
    ]
    inputs = tokenizer(render_messages(messages), return_tensors="pt").to(device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    new_tokens = output_ids[0, inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


results = []
for case in DIRECT_50_EVAL_CASES:
    response = generate_response(case)
    score = score_response(response, case)
    results.append({"case": asdict(case), "response": response, "score": asdict(score)})
len(results)

50

## Results

In [4]:
def flatten_result(index, result):
    case = result["case"]
    score = result["score"]
    return {
        "case_id": index,
        "expected_tool": case["expected_tool"],
        "prompt": case["prompt"],
        "expected_args": json.dumps(case["expected_args"], ensure_ascii=False),
        "response": result["response"],
        "valid_json": score["valid_json"],
        "json_only": score["json_only"],
        "tool_match": score["tool_match"],
        "args_match": score["args_match"],
        "total_reward": score["total_reward"],
    }

table_rows = [flatten_result(index, result) for index, result in enumerate(results, start=1)]
summary = {
    "adapter": str(ADAPTER_DIR),
    "cases": len(table_rows),
    "valid_json": sum(row["valid_json"] for row in table_rows),
    "json_only": sum(row["json_only"] for row in table_rows),
    "tool_match": sum(row["tool_match"] for row in table_rows),
    "args_match": sum(row["args_match"] for row in table_rows),
    "total_reward": sum(row["total_reward"] for row in table_rows),
    "max_reward": len(table_rows) * 4,
}
summary

{'adapter': '/home/shubeeksh/projects/toolcall-rl/outputs/sft_smollm_20tools',
 'cases': 50,
 'valid_json': 48,
 'json_only': 48,
 'tool_match': 41,
 'args_match': 21,
 'total_reward': 158,
 'max_reward': 200}

In [5]:
import pandas as pd

df = pd.DataFrame(table_rows)
display(df[["case_id", "expected_tool", "valid_json", "json_only", "tool_match", "args_match", "total_reward"]])

,case_id,expected_tool,valid_json,json_only,tool_match,args_match,total_reward
0,1,calculator,1,1,1,1,4
1,2,google_search,1,1,1,1,4
2,3,unit_converter,1,1,1,1,4
3,4,text_stats,1,1,1,1,4
4,5,string_formatter,1,1,1,1,4
5,6,weather_lookup,1,1,1,1,4
6,7,currency_converter,1,1,1,1,4
7,8,translate_text,1,1,1,1,4
8,9,create_calendar_event,1,1,1,1,4
9,10,send_email,1,1,1,1,4


## Save Results

In [6]:
jsonl_path = OUTPUT_DIR / "sft_50tools_direct_eval_results.jsonl"
csv_path = OUTPUT_DIR / "sft_50tools_direct_eval_results.csv"
with jsonl_path.open("w", encoding="utf-8") as file:
    for result in results:
        file.write(json.dumps(result, ensure_ascii=False) + "\n")
with csv_path.open("w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=table_rows[0].keys())
    writer.writeheader()
    writer.writerows(table_rows)
jsonl_path, csv_path

(PosixPath('/home/shubeeksh/projects/toolcall-rl/outputs/sft_50tools_direct_eval_results.jsonl'),
 PosixPath('/home/shubeeksh/projects/toolcall-rl/outputs/sft_50tools_direct_eval_results.csv'))